# 01 — Diagnose missing alkalinity in an atmosphere–ocean model

**Learning goals:** map boxes and arrows to a conserved carbon inventory;
diagnose why CO2 invasion cannot create TA; distinguish equilibrium controls
from rate controls.

This is a fictional redistribution experiment, not a history of ocean formation.
The first run has **TA = 0** and almost all carbon in the atmosphere.
Predict its outcome before running. Solutions are marked for masking in the
generated student copy.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import PyCO2SYS as pyco2

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'teaching_config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from teaching_config import TEACHING as C
from simple_models import (
    new_model, box_parameters, box_mass_kg, connect_atmosphere,
    single_box, two_layer, inventories, audit, mixing_mass_transport,
    calibrated_pump_coefficient, finite_box_addition, integrated_signal,
)
from esbmtk import (
    GasReservoir, Species2Species, Signal, Source,
    initialize_reservoirs, create_bulk_connections, add_carbonate_system_1,
)
from model import run_model

## 1. Specify the system before building it

```text
Atmosphere (xCO2; GasReservoir)
            ↕ gas exchange: carbon only (Species2Species)
Ocean (DIC, TA; initialize_reservoirs)
      carbonate system 1 → aqueous CO2
```

Both boxes lie inside the system boundary. Define gas flux $J_{gas}(t)$ positive
into the ocean. Then $dC_{atm}/dt=-J_{gas}(t)$ and
$m_o\,dDIC_o(t)/dt=J_{gas}(t)$; the TA inventory has zero tendency.

| Quantity | Status |
| --- | --- |
| 280 ppm dry-air xCO2; 2040 µmol/kg DIC | Observed comparison targets |
| Area, 3750 m ocean depth, atmospheric mole inventory, T/S/P | Independent inputs |
| Total carbon computed from those sizes and targets | Target-derived inventory |
| Initial carbon partition | Fictional initial condition |
| TA inferred later from the two targets | Fitted chemistry input |

Uniform T = 16 °C, S = 35, P = 0 bar and carbonate choices are supplied by
`teaching_config.py`. ESBMTK uses bar; PyCO2SYS uses dbar. Seawater density comes
from ESBMTK at these conditions. No geometry is fitted to a carbon ratio.

$$C_0=N_{atm}(280\times10^{-6})+\rho V_o(2040\times10^{-6}).$$

In [ ]:
print('Shared chemistry:', C.pyco2)
print('Ocean volume (m3):', C.ocean_volume_m3)
print('ESBMTK density (kg/m3):', C.density_kg_m3)
print('Target-derived total carbon (mol):', C.total_carbon_mol)
INITIAL_DIC = 0.01  # umol/kg, small positive numerical seed
FIRST_TA = 0.0      # umol/kg: no background alkalinity

## 2. Supplied example: translate the diagram into objects

`Model` supplies the clock and units. `initialize_reservoirs` creates DIC and TA
states; `add_carbonate_system_1` supplies aqueous CO2 for gas exchange.
The atmospheric mole fraction is calculated from the carbon remaining after
initializing the ocean. Gas exchange is bidirectional even though its positive
direction is atmosphere to ocean.

| Diagram part | ESBMTK object | Units or role |
| --- | --- | --- |
| Clock and chemistry choices | `Model` via `new_model` | years; mol; mol/kg |
| Ocean DIC and TA states | `M.Ocean.DIC`, `M.Ocean.TA` | mol/kg |
| Ocean geometry | `box_parameters` | m3, m2; mass = volume × density |
| Aqueous CO2 | `add_carbonate_system_1` | mol/kg; derived from DIC and TA |
| Atmosphere | `GasReservoir` inside `connect_atmosphere` | mole fraction; explicit air inventory |
| Bidirectional gas arrow | `Species2Species`, `ctype='gasexchange'` | mol C/yr; equal and opposite box tendencies |


In [ ]:
M = new_model(stop='2 kyr', max_timestep='1 yr')
initialize_reservoirs(M, {'Ocean': box_parameters(
    M, C.ocean_volume_m3, INITIAL_DIC, FIRST_TA)})
add_carbonate_system_1([M.Ocean])
# Supplied helper creates GasReservoir and the native gasexchange connection.
# Inspect its short definition in simple_models.py alongside this table.
exchange = connect_atmosphere(M, [M.Ocean])
assert exchange.source is M.CO2_At and exchange.sink is M.Ocean.DIC
np.testing.assert_allclose(box_mass_kg(M.Ocean),
                           C.ocean_volume_m3 * C.density_kg_m3, rtol=1e-12)
assert M.Ocean.TA.c[0] == 0
print('Initial atmosphere (ppm):', M.CO2_At.c[0] * 1e6)
print('Initial ocean carbon fraction:',
      box_mass_kg(M.Ocean) * M.Ocean.DIC.c[0] / C.total_carbon_mol)

### Predict, run, then diagnose

Can this model match both targets? What could cause TA to change inside this
closed diagram? Write your prediction before the next cell.

> **Your explanation:** replace this placeholder with your answer.

In [ ]:
run_model(M)
print(audit(M))
print('TA-free result: xCO2 (ppm), DIC (umol/kg):',
      M.CO2_At.c[-1] * 1e6, M.Ocean.DIC.c[-1] * 1e6)
assert abs(M.CO2_At.c[-1] * 1e6 - C.target_xco2_ppm) > 1000
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].plot(M.time, M.CO2_At.c * 1e6)
axes[0].axhline(C.target_xco2_ppm, ls='--', color='gray', label='observed target')
axes[0].set_ylabel('xCO2 (ppm)')
axes[0].legend()
axes[1].plot(M.time, M.Ocean.DIC.c * 1e6)
axes[1].axhline(C.target_dic_umol_kg, ls='--', color='gray')
axes[1].set(xlabel='Time (yr)', ylabel='DIC (umol/kg)')
fig.tight_layout()
plt.show()

## 3. Exercise: infer the missing TA, then check the implementation

Only now use PyCO2SYS with target DIC and dry-air xCO2 to infer TA under the
shared conditions. Keep the same total carbon, geometry, initial DIC and piston
velocity for the buffered rerun. This is a **calibration and software-consistency
check**, not an independent prediction of TA or atmospheric xCO2.

In [ ]:
raise NotImplementedError("Exercise: replace this line with your solution")
buffered = single_box(ta_umol_kg=inferred_ta)
run_model(buffered)
print('Inferred TA (umol/kg):', inferred_ta)
print(audit(buffered))
np.testing.assert_allclose(buffered.CO2_At.c[-1] * 1e6, 280, atol=0.5)
np.testing.assert_allclose(buffered.Ocean.DIC.c[-1] * 1e6, 2040, atol=0.2)

## 4. Exercise: change the path while retaining the inventory

Compare a second initial partition (1000 µmol/kg ocean DIC) at the same total
carbon and inferred TA. Then halve piston velocity using the original
near-empty ocean partition. Predict which endpoints should agree and which
paths should differ. Calculate inventories at every stored time, not just at
the endpoint. Concentration in mol/kg times water mass in kg gives moles;
atmospheric mole fraction times atmospheric moles gives moles of carbon.

In [ ]:
partition = single_box(ta_umol_kg=inferred_ta, initial_dic_umol_kg=1000)
slower = single_box(ta_umol_kg=inferred_ta, piston_velocity='2 m/d')
for case in (partition, slower):
    run_model(case)
    raise NotImplementedError("Exercise: replace this line with your solution")
    print(audit(case))

def settling_time(case):
    # First stored time after which xCO2 stays within 1% of its final value.
    x = case.CO2_At.c
    outside = np.flatnonzero(abs(x - x[-1]) > 0.01 * abs(x[-1]))
    return case.time[outside[-1] + 1] if len(outside) else case.time[0]

fig, ax = plt.subplots(figsize=(7, 4))
for label, case in [('near-empty ocean', buffered),
                    ('alternative partition', partition), ('half piston velocity', slower)]:
    ax.semilogy(case.time, case.CO2_At.c * 1e6, label=label)
    print(label, 'settling time (yr):', settling_time(case))
ax.set(xlim=(0, 300), xlabel='Time (yr)', ylabel='xCO2 (ppm)')
ax.legend()
plt.show()
assert settling_time(slower) > settling_time(buffered)

## 5. Explain the model boundary

Why do the equilibria agree although the paths differ? Where does real ocean
TA come from, and why is that not simulated here?

> **Your explanation:** replace this placeholder with your answer.

**Numerical note.** A 0.01 µmol/kg DIC seed is feasible with TA exactly zero;
it is not hidden buffering. ESBMTK's carbonate-system-1 approximation can show
transient pH discrepancies in extreme states (paper section 2.4). This exercise
checks carbon/TA conservation and stationary agreement; do not interpret its
extreme transient pH as a realistic seawater history. The gas-exchange helper
uses PyCO2SYS's dry-air conversion and ESBMTK density consistently, including
the native gas-exchange routine's factor-of-1000 convention.